# uobench Notebook 实操指南

本 Notebook 演示如何在 Python 环境中：
1. 生成带有可行性证书与诊断信息的 S 规模示例数据集；
2. 检查 `meta.json` / `data.npz` 并核验证书；
3. 调用基线求解器并绘制目标值与约束违反度曲线；
4. 导出包含维度、旋钮、诊断指标的 Markdown/CSV/JSON 报告。

所有代码单元都包含注释，解释参数含义与返回结果的结构，便于数学研究人员进一步定制。


## 1. 环境与基础导入
确保已在当前解释器中安装 `numpy`、`scipy`、`matplotlib`。以下单元加载核心模块，并设定数据输出目录。


In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np

from uobench.core.spec import PROBLEM_REGISTRY, SUITE_SPECS
from uobench.core import diagnostics, report, witness
from uobench.io import save_instance, load_instance
from uobench.solvers import gd, alm, prox
from uobench.utils.rng import RNG

# 所有示例实例将写入 datasets/notebook_demo 目录
DATA_ROOT = Path('datasets/notebook_demo')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT


## 2. 生成一组示例实例
我们选取 `core18` 套件中的五个代表性问题，逐一调用注册的生成器。每个参数都在注释中解释：
- `knobs`：指定规模/条件数/SNR 等难度旋钮；
- `seed`：通过 `RNG.spawn` 衍生，保证多次运行仍可复现；
- `diagnostics`：调用 `uobench.core.diagnostics.compute` 得到的难度指标；
- `save_instance`：写出 `meta.json`、`data.npz`、`README.md` 三个文件。


In [ ]:
suite_name = 'core18'  # 预置套件名称
suite_spec = SUITE_SPECS[suite_name]
target_problems = ['A1_QP', 'A4_ECQP', 'B1_LASSO', 'C2_LCP', 'D2_BP']
base_rng = RNG(42)  # 全局随机数生成器，负责派生每个问题的 seed

generated_paths = []
for idx, problem_id in enumerate(target_problems):
    spec = PROBLEM_REGISTRY[problem_id]
    scale_entry = suite_spec['problems'][problem_id]['S']  # 选用 S 规模配置
    knobs = dict(scale_entry['knobs'][0])  # 拷贝一份旋钮参数，便于手动修改
    seed = base_rng.spawn(idx).seed  # spawn() 确保不同问题拥有独立且可复现的随机种子

    instance = spec.generator(seed=seed, knobs=knobs, extreme=False)
    arrays = instance['data']  # 各类矩阵/向量存放于此

    meta = {
        'id': spec.problem_id,
        'name': spec.name,
        'family': spec.family,
        'seed': seed,
        'dims': instance['dims'],  # 记录变量/约束维度
        'knobs': instance['knobs'],  # 实际使用的旋钮值（可能由 extreme 模式改写）
        'witness': instance['witness'],  # 可行性证书（例如可行点、半径、互补对等）
        'diagnostics': diagnostics.compute(spec.problem_id, arrays),
        'reference': {'has_reference': bool(instance.get('reference'))},
    }
    readme_text = instance.get('readme', 'Instance for {} with knobs {}'.format(spec.problem_id, knobs))
    tag = 'seed_{:04d}'.format(seed)

    paths = save_instance(DATA_ROOT, suite_name, spec.problem_id, 'S', tag, meta, arrays, readme_text)
    generated_paths.append(paths)
    print(spec.problem_id, 'saved to', paths.meta.parent)

len(generated_paths)


## 3. 查看 metadata 并验证证书
此处演示如何加载单个实例、阅读存储的字段，并调用 `witness.verify` 检查证书（例如等式约束、非负性、互补条件）。


In [ ]:
instance_dirs = [paths.meta.parent for paths in generated_paths]

for inst_dir in instance_dirs:
    meta, arrays = load_instance(inst_dir)
    ok = witness.verify(meta['id'], meta, arrays)
    print(meta['id'], '| seed =', meta['seed'], '| dims =', meta['dims'], '| feasible =', ok)
    pprint(meta['diagnostics'])
    print('-' * 60)


## 4. 调用基线求解器并绘图
每个求解器都提供 `plot` 开关：开启后会绘制目标值（`history['f']`）与约束违反度（`history['constraint']`）的迭代曲线。


In [ ]:
# 4.1 A1_QP：梯度下降
a1_dir = instance_dirs[0]
meta_a1, arrays_a1 = load_instance(a1_dir)
res_gd = gd.solve_gd(meta_a1['id'], arrays_a1, max_iter=200, tol=1e-7, plot=True)
print('GD status:', res_gd['status'], 'iterations:', res_gd['iters'], 'final obj:', res_gd['obj'])

# 4.2 A4_ECQP：增广拉格朗日（自动绘制原始残差的 ∞ 范数）
a4_dir = instance_dirs[1]
meta_a4, arrays_a4 = load_instance(a4_dir)
res_alm = alm.solve_alm(meta_a4['id'], arrays_a4, max_iter=50, tol=1e-6, plot=True)
print('ALM status:', res_alm['status'], 'iterations:', res_alm['iters'])

# 4.3 B1_LASSO：FISTA
b1_dir = instance_dirs[2]
meta_b1, arrays_b1 = load_instance(b1_dir)
res_fista = prox.solve_fista(meta_b1['id'], arrays_b1, max_iter=200, tol=1e-6, plot=True)
print('FISTA status:', res_fista['status'], 'final objective:', res_fista['obj'])

# 4.4 C2_LCP：投影梯度法（投影到非负正交体）
c2_dir = instance_dirs[3]
meta_c2, arrays_c2 = load_instance(c2_dir)
res_pg = prox.solve_projected_gd(meta_c2['id'], arrays_c2, max_iter=200, tol=1e-6, plot=True)
print('Projected GD status:', res_pg['status'], 'final residual:', res_pg['obj'])

# 4.5 D2_BP：FISTA 解决等式稀疏问题
d2_dir = instance_dirs[4]
meta_d2, arrays_d2 = load_instance(d2_dir)
res_bp = prox.solve_fista(meta_d2['id'], arrays_d2, max_iter=200, tol=1e-6, plot=True)
print('Basis pursuit FISTA status:', res_bp['status'])


## 5. 汇总报告（Markdown / CSV / JSON）
使用 `report.summarize_instances` 可快速收集所有实例的核心字段。下方示例同时写出三种格式，便于论文附录或调试记录。


In [ ]:
paths = [Path(p.meta.parent) for p in generated_paths]
rows = report.summarize_instances(paths)

report_dir = Path('reports/notebook_demo')
report.write_markdown(report_dir / 'summary.md', rows)
report.write_csv(report_dir / 'summary.csv', rows)
report.write_json(report_dir / 'summary.json', rows)

print('Markdown report saved to', report_dir / 'summary.md')
print('First row preview:')
pprint(rows[0])


## 6. 重新加载并自定义参数
以下示例展示如何读取单个实例后修改 `knobs`/`seed`，重新生成带有新参数的数据集。


In [ ]:
meta_example, arrays_example = load_instance(instance_dirs[0])
print('Original knobs:', meta_example['knobs'])

# 自定义旋钮：例如提高条件数并重新生成 A1_QP
custom_knobs = dict(meta_example['knobs'])
custom_knobs['kappa'] = 1e5  # 目标更高的条件数
spec = PROBLEM_REGISTRY[meta_example['id']]
custom_instance = spec.generator(seed=1234, knobs=custom_knobs, extreme=True)
print('Customized diagnostics:', diagnostics.compute(spec.problem_id, custom_instance['data']))
